In [1]:
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_mistralai import MistralAIEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()

PDF_PATH = Path("../documents/hr_policy.pdf")

loader = PyPDFLoader(str(PDF_PATH))
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)

embeddings = MistralAIEmbeddings(
    model="mistral-embed"
)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)

print(f"Vectors stored: {vector_store.index.ntotal}")

/var/folders/f_/1j85lpk92ddfd4c7r8d8rz880000gn/T/ipykernel_6593/3523968244.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Vectors stored: 70


In [10]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 2}
    )
print(type(retriever))

<class 'langchain_core.vectorstores.base.VectorStoreRetriever'>


In [11]:
query = "How many paid leaves can an employee take?"

results = retriever.invoke(query)

print(f"Retrieved documents: {len(results)}")

for i, doc in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)

Retrieved documents: 2

--- Result 1 ---
ii. Total 12 days of Earned leaves will be granted per year.(Year considered as financial year from April- 
March) 
iii. Employee will get Earned leaves once they get complete probation period, it will be based on 
financial year. 
iv. The employer has rights to terminate Employee(s) without notice and without any payment in lieu  
of notice if Employee is taking more than 13 unplanned leaves (including weekends and official offs) 
without any proper updates (and proofs).

--- Result 2 ---
given time should not be less than 2:3. 
14. Employees going on leave have to explain/delegate their responsibility to another team member. 
15. In case an employee is extending the sanctioned leave, they will have to apply in writing with 
reason/valid proof and obtain permission before expiry of the leave originally granted to them. 
16. Maximum of 10 days Leave would be clubbed together, including official offs, however ELs shall be 
deducted only for the w

In [12]:
query = "What are the average working hours?"

results = retriever.invoke(query)

for i, doc in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)


--- Result 1 ---
deducted only for the working days. 
17. All the Saturdays of the month are off except Last Saturday of month (Training and development 
Saturdays) which will be counted as Working Day and employee has to attend minimum 7 hours to  
count it as a full day. However, the working hours of the last Saturday will not be counted in Monthly  
working hour’s average. 
18. Leaves application should be done at least 4 days before the leave date

--- Result 2 ---
 In case, the average monthly hours are not attained 8:40, the shortfall of hours will be rounded off to the 
nearest number of days (or half day) and equivalent number of days (or half day) will be considered as 
leaves for deduction.
 The following details would be considered for deduction if it lacks average monthly 8:40 hours:
 
 
Range of Mins / Hours Deductible Leaves 
Upto 15 Minutes Half Day (0.5) 
16 to 30 Minutes 1.5 Full Day 
31 to 45 Minutes 2.5 Full Day


In [13]:
queries = [
    "What is the leave policy?",
    "What are the working hours?",
    "What is the work from home policy?",
]

for query in queries:
    print("\n" + "=" * 80)
    print("QUERY:", query)

    results = retriever.invoke(query)

    for i, doc in enumerate(results, start=1):
        print(f"\nResult {i}")
        print(doc.page_content[:300])


QUERY: What is the leave policy?

Result 1
3. Leave Policy ............................................................................................................................................ 5 
3.1. Introduction ..........................................................................................................................

Result 2
31 to 45 Minutes 2.5 Full Day 
45 to 60 Minutes 3.5 Full Day 
1 Hour to 1.15 + Hours 4 Full Day and for each 15 Mins +additional 1 Day 
3. Leave Policy 
3.1. Introduction 
 Leave is a privilege extended by the Company to its staff members, and not a right. It shall be the  
Company’s discretion to 

QUERY: What are the working hours?

Result 1
deducted only for the working days. 
17. All the Saturdays of the month are off except Last Saturday of month (Training and development 
Saturdays) which will be counted as Working Day and employee has to attend minimum 7 hours to  
count it as a full day. However, the working hours of the last Sa

## Observation

A retriever provides a standardized interface for retrieving relevant
documents from a knowledge source.

A FAISS vector store can be converted into a LangChain retriever using
`as_retriever()`.

The retriever can then be queried using `invoke(query)` and returns relevant
LangChain `Document` objects.

The retriever is responsible for finding relevant information; it does not
generate the final answer. The retrieved documents will later be passed to
an LLM during the generation stage of the RAG pipeline.

Compared with directly calling `similarity_search()` on the vector store,
the retriever provides a cleaner abstraction between the application and
the underlying retrieval implementation.